In [7]:
"""
Feature Set Comparison: RandomShapelet | RDST | RandomInterval | SFA (Dictionary)
on 15 MTSC datasets.

Design: raw aeon TRANSFORMERS → 2D feature matrix → one shared RF classifier.
Fair comparison: every method uses RandomForestClassifier(n_estimators=100).
All parameters are set to publication-grade defaults (matching aeon/sktime
literature benchmarks and original paper settings).

Feature families:
  1. shapelet  – RandomShapeletTransform   (classical shapelet, 2017)
                   n_shapelet_samples=10_000 (aeon default / literature standard)
                   max_shapelets=None        (auto: min(10×n_cases, 1000))
                   remove_self_similar=True
                   n_jobs=-1

  2. rdst      – RandomDilatedShapeletTransform  (modern SOTA shapelet, 2023)
                   max_shapelets=10_000      (aeon default)
                   proba_normalization=0.8   (80% z-norm / 20% raw — paper default)
                   multivariate-native (no channel-wise wrapper needed)
                   n_jobs=-1

  3. interval  – RandomIntervals / RandomIntervalTransformer
                   n_intervals=100           (summary stats: mean/std/slope/IQR…)
                   n_jobs=-1

  4. sfa       – sktime SFA (Schäfer & Högqvist 2012)
                   Full DFT + MCB (Multiple Coefficient Binning) discretisation
                   word_length=8, alphabet_size=4  (BOSS/WEASEL paper defaults)
                   binning_method='equi-depth'     (original MCB)
                   norm=True, anova=False           (z-norm windows, no leakage)
                   window_size=min(series_len, 12)  (adapted at fit time)
                   Fallback: numpy DFT (if sktime not installed — pip install sktime)

Per-fold protocol (zero leakage):
    trf = FACTORY[name]()           # fresh transformer each fold
    trf.fit(X_train, y_train)       # y_train required by supervised transformers
    X_train_t = trf.transform(X_train)
    X_val_t   = trf.transform(X_val)
    rf.fit(X_train_t, y_train)
    rf.predict(X_val_t)

Install:
    pip install aeon sktime
"""

# ── Auto-install ──────────────────────────────────────────────────────────────
import subprocess, sys
for _pkg in ["aeon", "sktime"]:
    try:
        __import__(_pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", _pkg])

import warnings
import time
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction import DictVectorizer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score

from aeon.datasets import load_classification

warnings.filterwarnings("ignore")


# ── aeon version detection ────────────────────────────────────────────────────
from packaging.version import Version
import aeon as _aeon
_AEON_V1 = Version(_aeon.__version__) >= Version("1.0.0")
print(f"aeon version: {_aeon.__version__}")


# ── Transformer imports with graceful fallbacks ───────────────────────────────
# 1a. Classical shapelet
from aeon.transformations.collection.shapelet_based import RandomShapeletTransform

# 1b. Dilated shapelet (RDST) — SOTA 2023, multivariate-native
try:
    from aeon.transformations.collection.shapelet_based import (
        RandomDilatedShapeletTransform,
    )
    _HAS_RDST = True
    print("RDST: using RandomDilatedShapeletTransform")
except ImportError:
    _HAS_RDST = False
    print("RDST: RandomDilatedShapeletTransform not available in this aeon build — "
          "will skip rdst (upgrade aeon to >= 1.0)")

# 2. Interval-based
# Confirmed from aeon 1.3.0: RandomIntervals is the standalone interval
# feature transformer (fit→transform→2D array). RandomIntervalTransformer
# does not exist in this version — it was a name used in older docs only.
# RandomIntervals extracts random-length, random-position, random-dimension
# intervals and computes summary statistics (mean, std, slope, IQR, etc.).
from aeon.transformations.collection.interval_based import RandomIntervals
_INTERVAL_CLASS = RandomIntervals
print("Interval: using RandomIntervals ✓ (confirmed aeon 1.3.0 class)")

# 3. SFA / Dictionary — sktime's proper SFA (full DFT + MCB discretisation)
# aeon 1.3.0 SFAFast has an unfixed internal slice-assignment bug.
# sktime (aeon's parent project) has a stable SFA transformer with the complete
# Schäfer 2012 algorithm: DFT + Multiple Coefficient Binning (MCB).
# Install: pip install sktime
_SFA_CLASS  = None
_SFA_LABEL  = None
_SFA_SOURCE = None
try:
    from sktime.transformations.panel.dictionary_based import SFA as _SkSFA
    _SFA_CLASS  = _SkSFA
    _SFA_LABEL  = "sktime.SFA"
    _SFA_SOURCE = "sktime"
    print("SFA: using sktime.transformations.panel.dictionary_based.SFA "
          "(full DFT + MCB — Schäfer 2012)")
except ImportError:
    print("SFA: sktime not installed — falling back to numpy DFT workaround.\n"
          "      Install with: pip install sktime")


# ── Datasets (15 MTSC benchmarks) ─────────────────────────────────────────────
DATASETS = [
    "ArticularyWordRecognition",
    "AtrialFibrillation",
    "BasicMotions",
    "ERing",
    "HandMovementDirection",
    "Handwriting",
    "Libras",
    "LSST",
    "NATOPS",
    "PenDigits",
    "RacketSports",
    "SelfRegulationSCP1",
    "SelfRegulationSCP2",
    "StandWalkJump",
    "UWaveGestureLibrary",
]

FEATURE_SETS = ["shapelet", "rdst", "interval", "sfa"] if _HAS_RDST else ["shapelet", "interval", "sfa"]
print(f"Feature sets to run: {FEATURE_SETS}")


# ── Transformer factories — one fresh instance per CV fold ────────────────────

def _make_shapelet():
    """
    Classical RandomShapeletTransform at the aeon default / literature-standard
    settings. n_shapelet_samples=10_000 matches the original paper evaluation
    and the aeon benchmark suite. max_shapelets=None lets aeon auto-set to
    min(10 × n_cases, 1000), which is the recommended default.
    """
    return RandomShapeletTransform(
        n_shapelet_samples=10_000,   # aeon default — matches published benchmarks
        max_shapelets=None,          # auto: min(10×n_cases, 1000)
        remove_self_similar=True,    # prunes redundant shapelets (always True in lit)
        n_jobs=-1,
        random_state=42,
    )


def _make_rdst():
    """
    RandomDilatedShapeletTransform (RDST) — Guilleme et al. 2023 SOTA.
    Adds dilation (multi-scale), mixed normalisation, and 3 features per
    shapelet (min-dist, argmin, shapelet-occurrence) vs 1 in the classical
    variant. Natively handles multivariate input.
    max_shapelets=10_000 and proba_normalization=0.8 are the paper defaults.
    """
    return RandomDilatedShapeletTransform(
        max_shapelets=10_000,        # paper / aeon default
        proba_normalization=0.8,     # 80% z-normalised, 20% raw — paper default
        n_jobs=-1,
        random_state=42,
    )


class _RandomIntervalsWrapper:
    """
    Thin wrapper around RandomIntervals that computes n_intervals=sqrt(T) at
    fit time. RandomIntervals requires an integer — it does not accept 'sqrt'
    as a string. sqrt(series_len) matches the per-tree interval count used in
    TimeSeriesForestClassifier and CanonicalIntervalForestClassifier.
    """

    def __init__(self):
        self._trf = None

    def fit(self, X, y=None):
        series_len  = X.shape[2]
        n_intervals = max(1, int(np.sqrt(series_len)))
        for kwargs in [
            dict(n_intervals=n_intervals, random_state=42, n_jobs=-1),
            dict(n_intervals=n_intervals, random_state=42),
            dict(random_state=42),
        ]:
            try:
                self._trf = _INTERVAL_CLASS(**kwargs)
                break
            except TypeError:
                continue
        self._trf.fit(X, y)
        return self

    def transform(self, X):
        return self._trf.transform(X)


def _make_interval():
    """
    RandomIntervals — the interval feature extractor underlying TSF, CIF and
    DrCIF classifiers in aeon. n_intervals=sqrt(series_len) is the literature
    standard, computed at fit time by _RandomIntervalsWrapper.
    """
    return _RandomIntervalsWrapper()


class SkSFAWrapper:
    """
    BOSS-protocol multi-window SFA transformer for multivariate time series.

    Implements the core feature extraction from BOSS (Bag-of-SFA-Symbols,
    Schäfer 2015) and MUSE (multivariate extension, Schäfer & Leser 2017):

      For each channel independently:
        For each window_size w in {min_window, min_window+step, ..., series_len}:
          1. Extract sliding windows of length w
          2. DFT — keep first word_length Fourier coefficients
          3. MCB binning — data-adaptive breakpoints from train fold only
          4. Count bag-of-words histogram for this window size
        Concatenate histograms across all window sizes → features for this channel
      Concatenate across all channels → final feature vector

    This multi-scale coverage is the source of BOSS/WEASEL's power. A single
    fixed window (as used in plain SFA) misses structure at other scales.

    Parameters (BOSS / WEASEL paper defaults):
      word_length    = 6              (reduced for multi-window to control dim)
      alphabet_size  = 4              (Schäfer 2012 / BOSS default)
      norm           = True           (z-normalise windows — standard in lit)
      binning_method = 'equi-depth'   (original MCB from Schäfer 2012)
      anova          = False          (unsupervised — no label leakage)
      min_window     = 10             (BOSS default minimum window size)
      n_windows      = 10             (number of window sizes to search)

    Uses sktime SFA internally (confirmed working in sktime 0.40.1).
    sktime SFA requires nested DataFrame input (dim_0 column of pd.Series).

    fit(X, y=None)  — X shape: (n_samples, n_channels, series_len)
    transform(X)    — returns dense 2D float64 array
    """

    _logged = False

    def __init__(self):
        self._sfas_by_chan   = []   # list of lists: [chan][window_idx] → SFA
        self._dvecs_by_chan  = []   # list of lists: [chan][window_idx] → DictVectorizer
        self._windows        = []   # window sizes used
        self._word_len       = 6
        self._n_chan         = None

    @staticmethod
    def _to_nested_df(X_2d):
        """(n_samples, n_timepoints) → sktime nested DataFrame."""
        return pd.DataFrame({
            "dim_0": [pd.Series(X_2d[i]) for i in range(len(X_2d))]
        })

    @staticmethod
    def _to_dicts(raw):
        """sktime SFA output (array of dicts) → list of dicts."""
        return [d if isinstance(d, dict) else {} for d in np.array(raw).ravel()]

    def fit(self, X, y=None):
        self._n_chan  = X.shape[1]
        series_len   = X.shape[2]

        # Build window size grid — BOSS searches from min_window to series_len.
        # min_window is capped at series_len to handle short series (e.g. PenDigits
        # has series_len=8, so min_window must not exceed 8).
        min_win      = min(max(4, series_len // 10), series_len)
        max_win      = series_len
        n_windows    = 10
        step         = max(1, (max_win - min_win) // max(1, n_windows - 1))
        self._windows = sorted(set(
            range(min_win, max_win + 1, step)
        ))[:n_windows]
        # Ensure at least one window
        if not self._windows:
            self._windows = [series_len]

        # word_length must be even and <= window_size for all windows
        # Use 6 (common BOSS setting) capped at smallest window // 2
        self._word_len = max(2, min(6, (self._windows[0] // 2) // 2 * 2))

        if not SkSFAWrapper._logged:
            print(
                f"\n      [sktime BOSS-SFA]  word_length={self._word_len}  "
                f"alphabet_size=4  norm=True  binning=equi-depth  "
                f"n_windows={len(self._windows)}  "
                f"window_range=[{self._windows[0]}..{self._windows[-1]}]  "
                f"n_channels={self._n_chan}  series_len={series_len}",
                flush=True,
            )
            SkSFAWrapper._logged = True

        self._sfas_by_chan  = []
        self._dvecs_by_chan = []

        for c in range(self._n_chan):
            X_c_2d = X[:, c, :]                      # (n, T)
            sfas_c, dvecs_c = [], []

            for w in self._windows:
                wl = max(2, min(self._word_len, (w // 2) // 2 * 2))
                X_c_df = self._to_nested_df(X_c_2d)
                sfa = _SFA_CLASS(
                    word_length               = wl,
                    alphabet_size             = 4,
                    window_size               = w,
                    norm                      = True,
                    binning_method            = "equi-depth",
                    anova                     = False,
                    bigrams                   = False,
                    return_pandas_data_series = False,
                    n_jobs                    = 1,     # parallelism at outer level
                )
                sfa.fit(X_c_df, y)
                raw = sfa.transform(X_c_df)
                dv  = DictVectorizer(sparse=False)
                dv.fit(self._to_dicts(raw))
                sfas_c.append(sfa)
                dvecs_c.append(dv)

            self._sfas_by_chan.append(sfas_c)
            self._dvecs_by_chan.append(dvecs_c)

        return self

    def transform(self, X):
        parts = []
        for c in range(self._n_chan):
            X_c_2d = X[:, c, :]
            for sfa, dv in zip(self._sfas_by_chan[c], self._dvecs_by_chan[c]):
                X_c_df = self._to_nested_df(X_c_2d)
                raw    = sfa.transform(X_c_df)
                feat   = dv.transform(self._to_dicts(raw))
                parts.append(feat.astype(float))
        return np.hstack(parts)


class NumpyDFTWrapper:
    """
    Fallback: pure-numpy sliding-window DFT features (used when sktime is not
    installed). Computes DFT coefficients but omits MCB discretisation.
    Install sktime for the proper SFA: pip install sktime
    """

    _logged = False

    def __init__(self):
        self._window_size = None
        self._word_length = None
        self._n_chan      = None

    def fit(self, X, y=None):
        series_len        = X.shape[2]
        self._n_chan      = X.shape[1]
        self._window_size = max(4, min(series_len, 16))
        self._word_length = self._window_size // 2
        if not NumpyDFTWrapper._logged:
            print(
                f"\n      [numpy-DFT fallback]  window={self._window_size}  "
                f"word_len={self._word_length}  "
                f"NOTE: install sktime for proper SFA with MCB",
                flush=True,
            )
            NumpyDFTWrapper._logged = True
        return self

    def transform(self, X):
        ws, wl = self._window_size, self._word_length
        feats = []
        for c in range(self._n_chan):
            wins      = np.lib.stride_tricks.sliding_window_view(X[:, c, :], ws, axis=1)
            mu        = wins.mean(axis=-1, keepdims=True)
            std       = wins.std(axis=-1, keepdims=True)
            std       = np.where(std < 1e-8, 1.0, std)
            coef      = np.fft.rfft((wins - mu) / std, axis=-1)[:, :, :wl].real
            feats.extend([coef.mean(axis=1), coef.std(axis=1)])
        return np.hstack(feats)


# Select SFA implementation
_ChannelWiseSFA = SkSFAWrapper if _SFA_CLASS is not None else NumpyDFTWrapper


def _make_sfa():
    return _ChannelWiseSFA()


FACTORIES = {
    "shapelet": _make_shapelet,
    "rdst":     _make_rdst if _HAS_RDST else _make_shapelet,  # fallback if unavailable
    "interval": _make_interval,
    "sfa":      _make_sfa,
}


# ── Cleaning helper ───────────────────────────────────────────────────────────
def clean(X_feat):
    """Cast to float64, densify sparse, median-impute NaN / Inf."""
    # Handle sparse matrices (SFA may return these)
    try:
        import scipy.sparse as sp
        if sp.issparse(X_feat):
            X_feat = X_feat.toarray()
    except ImportError:
        pass

    if not isinstance(X_feat, np.ndarray):
        X_feat = np.array(X_feat, dtype=float)
    X_feat = X_feat.astype(float)

    if not np.all(np.isfinite(X_feat)):
        X_feat = np.where(np.isfinite(X_feat), X_feat, np.nan)
        X_feat = SimpleImputer(strategy="median").fit_transform(X_feat)

    return X_feat


# ── CV — transformer fit inside loop, shared RF ───────────────────────────────
def evaluate_with_cv(name, X, y):
    """
    KFold(n_splits=5, shuffle=True, random_state=42).
    Each fold:
      - fresh transformer fitted on X_train only (zero leakage)
      - 2D feature matrices for train + val
      - RandomForestClassifier(n_estimators=100, random_state=42)
    Returns mean/std of accuracy and weighted-F1, plus n_features from fold 0.
    """
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    acc_scores, f1_scores_list = [], []
    n_features = -1

    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X)):
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]

        # Fresh transformer — fits only on train fold.
        # y_train is passed so supervised transformers (e.g. RandomShapeletTransform
        # which sets requires_y=True) work correctly; unsupervised transformers
        # accept and silently ignore the y argument.
        if name not in FACTORIES:
            raise KeyError(
                f"No factory registered for '{name}'. "
                f"Available: {list(FACTORIES.keys())}"
            )
        trf = FACTORIES[name]()
        trf.fit(X_train, y_train)
        X_train_t = clean(trf.transform(X_train))
        X_val_t   = clean(trf.transform(X_val))

        if fold_idx == 0:
            n_features = X_train_t.shape[1]

        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rf.fit(X_train_t, y_train)
        pred = rf.predict(X_val_t)

        acc_scores.append(accuracy_score(y_val, pred))
        f1_scores_list.append(f1_score(y_val, pred, average="weighted"))

    return (
        float(np.mean(acc_scores)),
        float(np.std(acc_scores)),
        float(np.mean(f1_scores_list)),
        float(np.std(f1_scores_list)),
        n_features,
    )


# ── Per-dataset runner ────────────────────────────────────────────────────────
def run_dataset(dataset_name):
    X, y = load_classification(dataset_name)
    n_samples, n_channels, series_len = X.shape
    n_classes = len(np.unique(y))
    print(f"  samples={n_samples}  channels={n_channels}  "
          f"length={series_len}  classes={n_classes}")

    metrics_row  = {"dataset": dataset_name}
    features_row = {"dataset": dataset_name}

    for name in FEATURE_SETS:
        try:
            print(f"    {name:<12} ...", end="  ", flush=True)
            t0 = time.perf_counter()
            acc_m, acc_s, f1_m, f1_s, n_feat = evaluate_with_cv(name, X, y)
            elapsed = time.perf_counter() - t0

            print(f"acc={acc_m:.4f}±{acc_s:.4f}  "
                  f"f1={f1_m:.4f}±{f1_s:.4f}  "
                  f"feat={n_feat}  t={elapsed:.1f}s")

            metrics_row[f"{name}_accuracy"] = f"{acc_m:.6f}±{acc_s:.6f}"
            metrics_row[f"{name}_f1"]       = f"{f1_m:.6f}±{f1_s:.6f}"
            features_row[f"{name}_n_features"] = n_feat

        except Exception as exc:
            print(f"ERROR: {exc}")
            metrics_row[f"{name}_accuracy"]    = "NaN"
            metrics_row[f"{name}_f1"]          = "NaN"
            features_row[f"{name}_n_features"] = 0

    return metrics_row, features_row


# ── Main ──────────────────────────────────────────────────────────────────────
def run_all():
    all_metrics  = {}
    all_features = {}

    for dataset in DATASETS:
        print(f"\n{'='*70}")
        print(f"Processing dataset: {dataset}")
        print(f"{'='*70}")
        try:
            metrics_row, features_row = run_dataset(dataset)
        except Exception as exc:
            print(f"  [SKIP] {dataset}: {exc}")
            metrics_row  = {"dataset": dataset}
            features_row = {"dataset": dataset}
            for name in FEATURE_SETS:
                metrics_row[f"{name}_accuracy"]    = "NaN"
                metrics_row[f"{name}_f1"]          = "NaN"
                features_row[f"{name}_n_features"] = 0

        all_metrics[dataset]  = metrics_row
        all_features[dataset] = features_row

    metrics_df  = pd.DataFrame.from_dict(all_metrics,  orient="index")
    features_df = pd.DataFrame.from_dict(all_features, orient="index")

    metrics_df = metrics_df.reindex(
        columns=["dataset"] + [c for n in FEATURE_SETS
                                for c in (f"{n}_accuracy", f"{n}_f1")],
        fill_value="NaN",
    )
    features_df = features_df.reindex(
        columns=["dataset"] + [f"{n}_n_features" for n in FEATURE_SETS],
        fill_value=0,
    )

    timestamp     = datetime.now().strftime("%Y%m%d_%H%M%S")
    metrics_file  = f"sota_metrics_results_{timestamp}.csv"
    features_file = f"sota_feature_counts_{timestamp}.csv"

    metrics_df.to_csv(metrics_file,  index=False)
    features_df.to_csv(features_file, index=False)

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print(f"Results saved to:")
    print(f"  Metrics       : {metrics_file}")
    print(f"  Feature counts: {features_file}")
    print(f"{'='*70}")
    print(f"\nTotal datasets attempted : {len(DATASETS)}")
    ok = sum(
        1 for row in all_metrics.values()
        if any(str(row.get(f"{n}_accuracy", "NaN")) != "NaN" for n in FEATURE_SETS)
    )
    print(f"Successfully processed   : {ok}")

    print("\nMean 5-fold CV accuracy per feature set:")
    for name in FEATURE_SETS:
        vals = []
        for v in metrics_df[f"{name}_accuracy"]:
            try:
                vals.append(float(str(v).split("±")[0]))
            except Exception:
                pass
        tag = f"({_SFA_LABEL})" if name == "sfa" else ""
        if vals:
            print(f"  {name:<12} {tag:<15}: {np.mean(vals):.4f}  (n={len(vals)} datasets)")

    print("\nFirst few rows of metrics:")
    print(metrics_df.head().to_string())
    print("\nFirst few rows of feature counts:")
    print(features_df.head().to_string())

    return metrics_df, features_df


if __name__ == "__main__":
    metrics_df, features_df = run_all()

aeon version: 1.3.0
RDST: using RandomDilatedShapeletTransform
Interval: using RandomIntervals ✓ (confirmed aeon 1.3.0 class)
SFA: using sktime.transformations.panel.dictionary_based.SFA (full DFT + MCB — Schäfer 2012)
Feature sets to run: ['shapelet', 'rdst', 'interval', 'sfa']

Processing dataset: ArticularyWordRecognition
  samples=575  channels=9  length=144  classes=25
    shapelet     ...  acc=0.9791±0.0070  f1=0.9794±0.0070  feat=981  t=493.6s
    rdst         ...  acc=0.9913±0.0095  f1=0.9901±0.0104  feat=30000  t=287.7s
    interval     ...  acc=0.9130±0.0264  f1=0.9141±0.0254  feat=84  t=4.0s
    sfa          ...  
      [sktime BOSS-SFA]  word_length=6  alphabet_size=4  norm=True  binning=equi-depth  n_windows=10  window_range=[14..140]  n_channels=9  series_len=144
acc=0.9722±0.0065  f1=0.9721±0.0066  feat=202923  t=223.4s

Processing dataset: AtrialFibrillation
  samples=30  channels=2  length=640  classes=3
    shapelet     ...  acc=0.1000±0.0816  f1=0.0873±0.0909  feat=1